# Personalização de LLMs para Geração Automatizada de Respostas em Atendimento ao Cliente.

# Objetivo
Desenvolver um chatbot de atendimento ao cliente utilizando um modelo de linguagem de código aberto (LLM), aplicando técnicas de Fine-Tuning com LoRA para adaptar o modelo à geração de respostas contextualizadas, de acordo com exemplos de conversas e interações entre clientes e atendentes.

# Metodologia
O projeto foi desenvolvido seguindo as seguintes etapas:

**Etapa 1** —  Preparação dos dados: Organização de um conjunto fictício de perguntas e respostas  para utilização durante o treinamento do modelo.

**Etapa 2** — Pré-processamento: Preparação dos textos e conversão das informações para o formato adequado ao modelo de linguagem.

**Etapa 3** — Configuração do modelo: Carregamento de um LLM pré-treinado (falcon-7b), por meio da plataforma Hugging Face, e configuração dos parâmetros necessários para o treinamento.

**Etapa 4** — Fine-Tuning com LoRA: Aplicação da técnica Low-Rank Adaptation (LoRA), permitindo adaptar o modelo com menor quantidade de parâmetros treináveis e menor consumo de memória.

**Etapa 5** — Treinamento: Divisão dos dados em conjuntos de treinamento e teste, seguida do ajuste fino do modelo.

**Etapa 6** — Avaliação: Análise das respostas geradas utilizando a métrica BLEU, comparando os resultados do modelo com as respostas de referência.

**Etapa 7** — Análise dos resultados: Verificação da capacidade de adaptação do modelo, considerando as limitações relacionadas ao volume reduzido de dados utilizados.

# Modelo de Perguntas e Respostas

In [1]:
import json

dados = {
    "perguntas": [
        {
            "pergunta": "Olá! Como faço para solicitar um cartão?",
            "resposta": "A solicitação do cartão deve ser realizada pela empresa contratante, por meio dos canais de atendimento disponíveis."
        },
        {
            "pergunta": "Meu cartão ainda não chegou. O que devo fazer?",
            "resposta": "Você pode consultar o status da entrega junto à equipe de atendimento, informando seus dados de identificação."
        },
        {
            "pergunta": "Como posso consultar o saldo do meu cartão?",
            "resposta": "O saldo pode ser consultado pelo aplicativo ou pelos canais digitais disponibilizados pela empresa."
        },
        {
            "pergunta": "Meu cartão foi bloqueado. Como faço para desbloquear?",
            "resposta": "Para verificar o motivo do bloqueio e solicitar o desbloqueio, entre em contato com a equipe de atendimento."
        },
        {
            "pergunta": "O que faço se perder meu cartão?",
            "resposta": "Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via."
        },
        {
            "pergunta": "Meu saldo está incorreto. Como resolver?",
            "resposta": "Nossa equipe precisa verificar o histórico de movimentações. Encaminhe sua solicitação ao atendimento informando os detalhes da divergência."
        },
        {
            "pergunta": "Quando será realizada a próxima recarga?",
            "resposta": "As datas de recarga dependem do calendário definido pela empresa contratante. Consulte o setor responsável ou os canais de atendimento."
        },
        {
            "pergunta": "Posso utilizar meu cartão em qualquer estabelecimento?",
            "resposta": "A utilização depende da modalidade do benefício, das regras de aceitação e dos estabelecimentos habilitados."
        },
        {
            "pergunta": "Meu cartão apresentou erro na compra. O que aconteceu?",
            "resposta": "A transação pode ter sido recusada por diferentes motivos, como saldo insuficiente, bloqueio ou restrições de utilização. O atendimento poderá verificar o ocorrido."
        },
        {
            "pergunta": "Como solicito uma segunda via?",
            "resposta": "Entre em contato com a equipe de atendimento para verificar as condições e os procedimentos para emissão de uma segunda via."
        },
        {
            "pergunta": "Quero falar com um atendente humano.",
            "resposta": "Claro! Vou direcionar sua solicitação para a equipe de atendimento humano."
        },
        {
            "pergunta": "Obrigado pela ajuda!",
            "resposta": "Por nada! Ficamos felizes em ajudar. Se precisar de mais alguma informação, estamos à disposição."
        }
    ]
}

with open("perguntas_respostas.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados, arquivo, ensure_ascii=False, indent=4)

#Instalando e Carregando os Pacotes

In [2]:
#Instala o pacote watermark
#Esse pacote é utilizado para gravar as versões de outros pacotes usados nesse jupyter notebook.
!pip install -q -U watermark

In [5]:
# Imports
import os             # manipular o sistema operacional
import json           # conjunto de dados nesse formato
import torch          # fazer o carregamento do modelo após o ajuste fino
import evaluate       # avaliar a performance do LLM após o ajuste fino
import torch.nn as nn  # neuronetwork (preparar o treinamento)
import transformers
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model
from datasets import Dataset, Features, ClassLabel, Value, Sequence
import warnings
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

# Definindo os Parâmetros de Quantização

In [6]:
#Definindo os parâmetros
quantization_config = BitsAndBytesConfig(load_in_4bit = True,
                                         bnb_4bit_compute_dtype = torch.float16,
                                         bnb_4bit_quant_type = "nf4",
                                         bnb_4bit_use_double_quant = True,
                                         llm_int8_enable_fp32_cpu_offload = True)

Os parâmetros acima são configurações específicas para a quantização de modelos de aprendizado de máquina com o PyTorch.

**load_in_4bit (True):** Este parâmetro indica que o modelo deve ser carregado em um formato de 4 bits. Isso é  definido para reduzir o uso de memória, permitindo que modelos grandes sejam carregados em hardware com memória limitada. Quando True, o modelo é carregado com uma precisão reduzida.

**bnb_4bit_compute_dtype (torch.float16):** Define o tipo de dados a ser usado para cálculos internos quando o modelo está em formato de 4 bits. O torch.float16 indica que os cálculos serão realizados usando o tipo de dados de ponto flutuante de 16 bits.

**bnb_4bit_quant_type ("nf4"):** Especifica o tipo de quantização a ser aplicada. O "nf4" se refere a um tipo específico de quantização de 4 bits. Referência: https://huggingface.co/blog/4bit-transformers-bitsandbytes

**bnb_4bit_use_double_quant (True):** Indica se deve ser usada uma "quantização dupla" durante o processo de quantização. A quantização dupla pode ajudar a manter mais precisão nos dados quantizados, mas pode ter um custo computacional mais alto.

**llm_int8_enable_fp32_cpu_offload (True):** Este parâmetro sugere que para operações de baixa latência (em modelos grandes como LLMs - Large Language Models), quando os cálculos estão sendo feitos em int8 (8 bits), há uma opção para descarregar (offload) algumas operações para o CPU em formato fp32 (ponto flutuante de 32 bits). Isso pode ser útil para equilibrar carga entre CPU e GPU e para lidar com operações que exigem maior precisão.
Referência: https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu

#Carregando  o Modelo e o Tokenizador

https://huggingface.co/tiiuae/falcon-7b

In [7]:
#Modelo
modelo = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-7b",
                                              quantization_config = quantization_config,
                                              device_map = 'auto')

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [8]:
#Tokenizador
tokenizador = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

# Congelando os Pesos Originais

In [9]:
#Loop
for param in modelo.parameters():
  param.requires_grad = False
  if param.ndim ==1:
    param.data = param.data.to(torch.float32)

**param.requires_grad = False:** Impede que os parâmetros originais do modelo sejam modificados durante o treinamento. Dessa forma, o modelo mantém os conhecimentos que já adquiriu anteriormente.

**param.ndim == 1:** Verifica se o parâmetro possui apenas uma dimensão, ou seja, se está organizado como um vetor.

**param.data.to(torch.float32):** Altera o formato numérico desses parâmetros para FP32, que oferece maior precisão nos cálculos.

**Objetivo dessa etapa:** Preservar os pesos originais do LLM e preparar o modelo para receber adaptações durante o fine-tuning com LoRA.

# Ativando o Checkpoint de Gradientes do Modelo

In [10]:
# Ativa o recurso de checkpoint de gradientes no modelo (Otimizar o uso de hardware)
modelo.gradient_checkpointing_enable()

**Gradient Checkpointing:** Quando essa técnica está ativada, o modelo deixa de armazenar todos os resultados intermediários gerados durante a execução, guardando apenas alguns deles. Durante o cálculo dos gradientes, os resultados que não foram armazenados são calculados novamente.

Isso ajuda a economizar memória da GPU, pois reduz a quantidade de informações que precisam ficar armazenadas durante o treinamento. Por outro lado, o processo pode demorar um pouco mais, já que algumas operações precisam ser refeitas.

Essa técnica é especialmente útil no treinamento de modelos grandes, que exigem muita memória da GPU.

In [11]:
# Habilita a técnica chamada "checkpointing de gradiente"
modelo.enable_input_require_grads()

# Ajustando a Conversão Para Tensor

In [12]:
#Conversão de tensor (Para ajustar o tipo de dados, para tentar aumentar a precisão, pois o volume de dados é pequeno")
class CastOutputToFloat(nn.Sequential):
  def foward(self, x): return super().foward(x).to(torch.float32)

Essa classe tem a função de converter a saída do modelo para o formato float32 (FP32). Essa conversão busca aumentar a precisão numérica dos resultados, especialmente quando o modelo utiliza tipos de dados com menor precisão, como FP16.

**nn.Sequential:** Permite organizar e executar camadas do modelo em sequência.



**forward(self, x):** Define como os dados serão processados durante a passagem pelo modelo.



**super().forward(x):** Executa as operações da classe original.



**.to(torch.float32):** Converte o tensor de saída para FP32.

In [13]:
modelo.lm_head = CastOutputToFloat(modelo.lm_head)

**Modelo.LM_head:** substitui a camada lm_head original por um módulo que mantém seu funcionamento, mas converte automaticamente os resultados de saída para torch.float32. Dessa forma, sempre que essa camada for executada, seus valores serão transformados para FP32, garantindo maior precisão numérica e compatibilidade com operações que exigem esse formato de dados.

#Parâmetros Para o Ajuste Fino do Modelo

In [14]:
#LoRa Config
config = LoraConfig(r = 16,
                    lora_alpha = 32,
                    lora_dropout = 0.05,
                    bias = "none",
                    task_type = "CAUSAL_LM")

Os parâmetros dentro de LoraConfig definem como o LoRA será aplicado ao modelo, controlando a quantidade de parâmetros treináveis, a intensidade das adaptações e outros aspectos do treinamento.

**r(Rank/ Dimensão de Classificação)**: Define o tamanho das matrizes adicionais utilizadas pelo LoRA. Um valor menor significa menos parâmetros para treinar e menor consumo de memória. Já um valor maior permite adaptações mais complexas, mas exige mais recursos computacionais.

**lora_alpha(Fator de Escala)**: Controla a intensidade das alterações realizadas pelo LoRA nos pesos originais do modelo. Valores maiores podem aumentar a influência das adaptações durante o treinamento.

**lora_dropout(Taxa de Drpout)**: Define a probabilidade de algumas conexões serem desativadas temporariamente durante o treinamento. Isso ajuda a evitar que o modelo fique muito dependente dos exemplos utilizados, melhorando sua capacidade de responder a novos dados.

**bias**: Define se os parâmetros de bias das camadas também serão ajustados durante o treinamento. Quando configurado como "none", nenhum bias adicional será treinado pelo LoRA.

**task_type**: Indica qual tipo de tarefa o modelo irá executar. O valor "CAUSAL_LM" identifica modelos de linguagem que geram textos sequencialmente, utilizando as palavras anteriores como contexto.

In [15]:
# Cria o modelo considerando os parâmetros LoRa
modelo = get_peft_model(modelo, config)

In [16]:
# Definindo a função para imprimir os parâmetros treináveis de um modelo
def trainable_parameters(model):

    # Inicializa a contagem de parâmetros treináveis
    trainable_params = 0

    # Inicializa a contagem de todos os parâmetros
    all_param = 0

    # Itera sobre todos os parâmetros nomeados do modelo
    for name, param in model.named_parameters():

        # Soma o número total de elementos de todos os parâmetros
        all_param += param.numel()

        # Verifica se o parâmetro é treinável
        if param.requires_grad:

            # Soma o número de elementos aos parâmetros treináveis
            trainable_params += param.numel()

    # Imprime o número de parâmetros treináveis, o total de parâmetros e a porcentagem de parâmetros treináveis
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

In [17]:
#Executa a função
trainable_parameters(modelo)

trainable params: 4718592 || all params: 3613463424 || trainable%: 0.13058363808693696


#Processamento dos Dados

In [18]:
#Tipo de dados
type(dados)

dict

In [19]:
#Visualisa os dados
dados

{'perguntas': [{'pergunta': 'Olá! Como faço para solicitar um cartão?',
   'resposta': 'A solicitação do cartão deve ser realizada pela empresa contratante, por meio dos canais de atendimento disponíveis.'},
  {'pergunta': 'Meu cartão ainda não chegou. O que devo fazer?',
   'resposta': 'Você pode consultar o status da entrega junto à equipe de atendimento, informando seus dados de identificação.'},
  {'pergunta': 'Como posso consultar o saldo do meu cartão?',
   'resposta': 'O saldo pode ser consultado pelo aplicativo ou pelos canais digitais disponibilizados pela empresa.'},
  {'pergunta': 'Meu cartão foi bloqueado. Como faço para desbloquear?',
   'resposta': 'Para verificar o motivo do bloqueio e solicitar o desbloqueio, entre em contato com a equipe de atendimento.'},
  {'pergunta': 'O que faço se perder meu cartão?',
   'resposta': 'Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.'}

In [20]:
#Listas para perguntas e respostas
perguntas = []
respostas = []


In [21]:
#Loop pelos dados para extrair perguntas e respostas

for i in dados["perguntas"]:
  perguntas += [i["pergunta"]]
  respostas += [i["resposta"]]

In [22]:
#Conteúdo do dataset original
dados["perguntas"][0]

{'pergunta': 'Olá! Como faço para solicitar um cartão?',
 'resposta': 'A solicitação do cartão deve ser realizada pela empresa contratante, por meio dos canais de atendimento disponíveis.'}

In [23]:
#Preparação dos dados no formato adequado para treinar o modelo
dataset = Dataset.from_dict({
    "id": list(range(len(perguntas))),
    "perguntas": perguntas,
    "respostas": respostas},
    features = Features({
        "id": Value(dtype = "string"),
        "perguntas": Value(dtype = "string"),
        "respostas": Value(dtype = "string")
    })

)

In [24]:
#Divide os dados em treino e teste
dataset = dataset.train_test_split(test_size = 0.15)

In [25]:
#Função para o merge de colunas concatenando cada pergunta com a resposta correspondente
def merge_columns(registro):
  registro["saida"] = registro["perguntas"] + " ->:" + registro["respostas"]

  return registro

In [26]:
#Aplica a função
novo_dataset = dataset.map(merge_columns)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [27]:
#Formato dos dados para treinar o modelo
novo_dataset ["train"] ["saida"][0]

'O que faço se perder meu cartão? ->:Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.'

In [28]:
#conjunto de treinamento
novo_dataset["train"][0]

{'id': '4',
 'perguntas': 'O que faço se perder meu cartão?',
 'respostas': 'Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.',
 'saida': 'O que faço se perder meu cartão? ->:Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.'}

In [29]:
#Tokenização dos dados
novo_dataset = novo_dataset.map(lambda samples: tokenizador(samples['saida']), batched = True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [30]:
#Dados tokenizados
novo_dataset["train"][0]

{'id': '4',
 'perguntas': 'O que faço se perder meu cartão?',
 'respostas': 'Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.',
 'saida': 'O que faço se perder meu cartão? ->:Entre em contato imediatamente com o atendimento para solicitar o bloqueio preventivo e verificar os procedimentos para emissão de uma segunda via.',
 'input_ids': [58,
  877,
  17752,
  90,
  385,
  47059,
  38140,
  4834,
  2682,
  42,
  204,
  1579,
  37,
  41052,
  732,
  537,
  1892,
  507,
  269,
  5132,
  4057,
  371,
  264,
  60654,
  10316,
  2179,
  22453,
  4805,
  264,
  23753,
  1020,
  752,
  3045,
  5299,
  293,
  1491,
  34620,
  5373,
  4005,
  45732,
  2179,
  732,
  740,
  2682,
  336,
  8569,
  35065,
  2666,
  25],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

# Definindo os argumentos de treino

In [31]:
#Garantir que o tokenizador tenha um token de preenchimento definido para organizar sequências de diferentes tamanhos durante o treinamento.
if tokenizador.pad_token == None:
  tokenizador.pad_token = tokenizador.eos_token

In [32]:
# Argumentos de treino
model_trainer = transformers.Trainer(model = modelo,
                                   train_dataset = novo_dataset["train"],
                                   eval_dataset = novo_dataset["test"],
                                   args = transformers.TrainingArguments(eval_strategy = "epoch",
                                                                         per_device_train_batch_size = 2,
                                                                         gradient_accumulation_steps = 2,
                                                                         num_train_epochs = 10,
                                                                         learning_rate = 2e-4,
                                                                         fp16 = True,
                                                                         logging_steps = 1,
                                                                         output_dir = 'outputs',
                                                                         report_to = "none"),
                                   data_collator = transformers.DataCollatorForLanguageModeling(tokenizador, mlm = False))

**model:** Define o modelo que será utilizado no treinamento.

**train_dataset:** Indica o conjunto de dados utilizado para ensinar o modelo

**eval_dataset:** Define os dados utilizados para avaliar o desempenho do modelo durante o treinamento

**args:** Reúne as configurações que determinam como o treinamento será realizado.

**evaluation_strategy = "epoch":** Realiza uma avaliação ao final de cada época.

**per_device_train_batch_size = 2:** Define que cada lote de treinamento terá 2 exemplos por dispositivo.

**gradient_accumulation_steps = 2:** Acumula os gradientes de duas etapas antes de atualizar os parâmetros do modelo.

**num_train_epochs = 10:** Define que o modelo passará 10 vezes pelos dados de treinamento.

**learning_rate = 2e-4:** Define a velocidade com que os parâmetros treináveis serão ajustados.

**fp16 = True:** Utiliza precisão de 16 bits para reduzir o consumo de memória e acelerar o treinamento em GPUs compatíveis.

**logging_steps = 1:** Registra informações do treinamento a cada etapa.

**output_dir = 'outputs':** Define a pasta onde os resultados do treinamento serão salvos.

**data_collator:** Organiza os exemplos em lotes, preparando os dados para serem processados pelo modelo.

**DataCollatorForLanguageModeling:** Prepara os dados para o treinamento de linguagem.

- tokenizer: Utiliza o tokenizador para organizar os textos.
- mlm = False: Indica que o modelo utilizará modelagem de linguagem causal, prevendo os próximos tokens com base no contexto anterior.

# Treinamento do Modelo

In [33]:
#Não usa o cache
modelo.config.use_cache = False

In [34]:
#Treinamento e ajuste fino do modelo
model_trainer.train()

{'loss': 2.7804, 'grad_norm': 0.5493615865707397, 'learning_rate': 0.0002, 'epoch': 0.4}
{'loss': 2.7903, 'grad_norm': 0.8087655305862427, 'learning_rate': 0.00019333333333333333, 'epoch': 0.8}
{'loss': 2.6413, 'grad_norm': 0.8496158123016357, 'learning_rate': 0.0001866666666666667, 'epoch': 1.0}
{'eval_loss': 2.2853622436523438, 'eval_runtime': 0.22, 'eval_samples_per_second': 9.091, 'eval_steps_per_second': 4.545, 'epoch': 1.0}
{'loss': 2.6048, 'grad_norm': 1.162939429283142, 'learning_rate': 0.00018, 'epoch': 1.4}
{'loss': 2.7808, 'grad_norm': 0.9251179099082947, 'learning_rate': 0.00017333333333333334, 'epoch': 1.8}
{'loss': 2.5872, 'grad_norm': 0.9580712914466858, 'learning_rate': 0.0001666666666666667, 'epoch': 2.0}
{'eval_loss': 2.2149930000305176, 'eval_runtime': 0.268, 'eval_samples_per_second': 7.463, 'eval_steps_per_second': 3.732, 'epoch': 2.0}
{'loss': 2.4672, 'grad_norm': 0.913398802280426, 'learning_rate': 0.00016, 'epoch': 2.4}
{'loss': 2.508, 'grad_norm': 1.15431356430

TrainOutput(global_step=30, training_loss=2.1705448706944783, metrics={'train_runtime': 42.4416, 'train_samples_per_second': 2.356, 'train_steps_per_second': 0.707, 'train_loss': 2.1705448706944783, 'epoch': 10.0})

# Avaliação do Modelo

In [35]:
# Função para extrair as previsões
def model_predict(question):

    # Garante que não estamos em modo treino
    modelo.eval()

    # Identifica cpu ou cuda (GPU)
    device = next(modelo.parameters()).device

    # tokeniza e já envia para o mesmo dispositivo do modelo
    batch = tokenizador(f"{question} ->: ",
                        return_tensors='pt',
                        padding=True,
                        truncation=True)

    batch = {k: v.to(device) for k, v in batch.items()}

    # Sem gradiente e com autocast em GPU para fp16
    with torch.no_grad(), torch.cuda.amp.autocast():
        output_tokens = modelo.generate(
            **batch,
            max_new_tokens=50,
            pad_token_id=tokenizador.eos_token_id
        )

    # Decodifica de volta para string
    return tokenizador.decode(output_tokens[0], skip_special_tokens = True) #Retorna para o formato texto

In [36]:
#Lista para as previsões
previsoes = []

In [37]:
#Loop para extrair as previsões
for i in novo_dataset["test"]["perguntas"]:
  previsoes.append(model_predict(i))

#Performance

In [38]:
#Carrega o módulo da métrica
bleu = evaluate.load('bleu')

A métrica BLEU é utilizada em Processamento de Linguagem Natural (PLN) para avaliar a qualidade dos textos gerados por modelos de linguagem. Ela compara a resposta produzida pelo modelo com uma ou mais respostas de referência.

Seu funcionamento considera a semelhança entre palavras e sequências de palavras presentes nos textos.

Limitação: O BLEU avalia principalmente a correspondência entre palavras e expressões. Por isso, pode apresentar pontuação baixa mesmo quando uma resposta possui o mesmo significado da referência, mas utiliza palavras diferentes.

In [39]:
#Extrai os dados reais
dados_reais = novo_dataset["test"]["saida"]

In [40]:
#Calcula a métrica comparandos os valores reais e as previsões
resultado = bleu.compute(predictions = previsoes, references = dados_reais)

In [41]:
resultado

{'bleu': 0.2796626360219556,
 'precisions': [0.38202247191011235,
  0.2988505747126437,
  0.24705882352941178,
  0.21686746987951808],
 'brevity_penalty': 1.0,
 'length_ratio': 1.390625,
 'translation_length': 89,
 'reference_length': 64}

**Bleu:** Esta é a pontuação BLEU geral do texto traduzido em relação ao texto de referência. A pontuação varia de 0 a 1, onde valores mais altos indicam melhor correspondência com o texto de referência.

**precisions:** Estas são as precisões dos n-gramas de 1 a 4-gramas, respectivamente. A precisão para unigramas é 0.38, indicando que 38% dos unigramas na tradução coincidem com aqueles nos textos de referência.

**brevity_penalty:** A penalidade por brevidade (BP) ajusta a pontuação BLEU para traduções que são significativamente mais curtas que seus textos de referência. Uma BP próxima de 1, como neste caso, indica que a diferença de comprimento entre a tradução e o texto de referência é mínima, portanto, a penalidade aplicada é baixa.

**length_ratio:** Este é o ratio entre o comprimento da tradução e o comprimento do texto de referência. Um valor de aproximadamente 1.39 sugere que o comprimento da tradução é quase igual ao do texto de referência, corroborando a baixa brevidade da penalidade.

**translation_length:** O comprimento total da tradução gerada, medido em número de palavras.

**reference_length:** O comprimento total do texto de referência, também em número de palavras. A proximidade entre os comprimentos da tradução e da referência confirma que a penalidade por brevidade foi mínima.

Apesar do baixo volume de dados utilizado durante o treinamento, o modelo conseguiu aprender padrões presentes nas perguntas e respostas, demonstrando capacidade de adaptação à tarefa proposta. O resultado BLEU de aproximadamente 28% indica uma correspondência moderada entre as respostas geradas e as referências utilizadas.

É importante considerar que a quantidade limitada de exemplos pode ter restringido o aprendizado e a capacidade de generalização do modelo. Ainda assim, o experimento demonstrou que, mesmo com poucos dados, foi possível realizar o fine-tuning e obter respostas geradas a partir dos padrões aprendidos.

Como possibilidade de melhoria, recomenda-se ampliar o conjunto de dados e realizar novos treinamentos, buscando melhorar a qualidade, a coerência e a precisão das respostas.

#Deploy e Uso do Modelo

In [42]:
# Define o device
device = next(modelo.parameters()).device

In [63]:
nova_pergunta_usuario = 'Posso utilizar meu cartão em qualquer estabelecimento?'

In [64]:
pergunta_tokenizada = tokenizador(nova_pergunta_usuario, return_tensors = "pt", padding = True, truncation = True)

In [65]:
# Move o batch para o mesmo dispositivo do modelo
pergunta_tokenizada = {nome: tensor.to(device) for nome, tensor in pergunta_tokenizada.items()}

**Autocast:** É um recurso do PyTorch que permite utilizar diferentes níveis de precisão durante o treinamento do modelo. Ele seleciona automaticamente quais operações podem ser executadas em FP16 e quais precisam permanecer em FP32.

Essa técnica ajuda a reduzir o consumo de memória da GPU e pode acelerar o treinamento, mantendo a precisão necessária para determinadas operações.

Para evitar problemas nos cálculos dos gradientes, o autocast pode ser utilizado em conjunto com o GradScaler, que ajusta os valores dos gradientes para reduzir perdas de precisão.

**Objetivo:** Tornar o treinamento mais rápido e econômico em memória, sem comprometer significativamente a estabilidade numérica do modelo.

In [70]:
# Gera a previsão sem cálculo de gradiente e com autocast
with torch.no_grad(), torch.cuda.amp.autocast():
    previsao_tokens = modelo.generate(**pergunta_tokenizada, max_new_tokens = 50, pad_token_id = tokenizador.eos_token_id)

In [71]:
#Decode de saída
tokenizador.decode(previsao_tokens[0], skip_special_tokens = True)

'Posso utilizar meu cartão em qualquer estabelecimento? -> Sim. O cartão pode ser utilizado em qualquer estabelecimento que aceite o tipo de pagamento.\nComo posso verificar o limite de crédito? -> O limite de crédito é informado no boleto mensal'

In [72]:
%reload_ext watermark

In [73]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.13.16
IPython version      : 7.34.0

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 6.6.159+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 2
Architecture: 64bit



In [74]:
%watermark --iversions

bitsandbytes: 0.50.2
datasets    : 4.0.0
evaluate    : 0.4.6
json        : 2.0.9
peft        : 0.16.0
torch       : 2.11.0+cu130
transformers: 4.54.0



# Conclusão


O projeto demonstrou a possibilidade de adaptar um modelo de linguagem de código aberto para uma aplicação de atendimento ao cliente utilizando a técnica de Fine-Tuning com LoRA. Mesmo com um volume reduzido de dados, o modelo conseguiu aprender padrões das conversas e gerar respostas com base nos exemplos apresentados durante o treinamento.

Os resultados também evidenciaram limitações relacionadas à quantidade de dados e à correspondência entre as respostas geradas e as referências. Como possibilidades de evolução, destacam-se a ampliação da base de treinamento, o aprimoramento dos parâmetros e a utilização de métricas complementares para avaliar a qualidade das respostas.